# Task 18 — Cleaning, Ranking, and the IPL Points Table

This notebook is a marathon practice worksheet across three very different real datasets. It is explicitly built around the full data-cleaning lifecycle: reading messy CSV files, inspecting them, patching holes, dropping duplicates and unwanted columns, converting dtypes, and restructuring indexes — and only after the data is clean, answering analytical questions on top of it.

The first dataset is the FIFA World Cup 2022 results table, a match-by-match record of the tournament. Q-1 computes, for every team, the percentage of total attempts that were on target, and builds a sorted Python dictionary out of it. Q-2 counts how many matches each team played and then ranks the teams, exploring the `rank()` method and its tie-breaking `method='first'` parameter. Q-3 is the classic cleaning pass: `info()`, `describe()`, missing-value counts, filling holes with the column average, removing duplicate rows permanently, and dropping a listed set of columns. Q-4 builds a `Rank` column, converts its dtype with `np.int16`, promotes the column to the index, and sorts by it.

The second dataset is the famous Titanic competition data. Q-5 practises the two branches of missing-value handling on two different files: dropping records that have missing Age values on the first file, and filling the Age column with the constant 20 on the second.

The third dataset is the IPL match history, and here the difficulty rises sharply. Q-6 builds a full per-team performance frame — matches played, win percentage, home win percentage, away win percentage — after renaming the three teams that changed names. Q-7 finds the venues with the most 'no result' matches. Q-8 counts player appearances in finals by parsing the string-valued `Team1Players` and `Team2Players` columns. Q-9 writes a complete `point_table(season)` function that builds an IPL points table with the correct scoring rules. Q-10 extends that table with a `SeasonPosition` column, labelling the Winner, Runner, Qualifier-2 loser, and Eliminator loser for each year — the kind of end-to-end task that appears in real analyses.

Work strictly in order: the cleaning skills from Q-3 and Q-5 are assumed by every later question. When you finish a question, print the result and compare it with the sample outputs — the numbers are your verification. There is no single correct style of answer; any correct pandas code is acceptable, and the goal is fluency with the cleaning and aggregation pipeline as a whole.

In [ ]:
import numpy as np
import pandas as pd

# to visualize all the column, run the below code
pd.options.display.max_columns = None
# to show all the records, use the below code
# pd.options.display.max_rows = None

# 1. The Fifa World Cup 2022 Dataset

## 1.1 Reading data straight from a URL

The World Cup 2022 results dataset is a row-per-match event table: each row records one team's performance in one of the tournament's matches, with columns such as Team, Against, Goals For, Goals Against, Attempts, On Target, Yellow Cards, Red Cards, Passes, Possession, and Points. The full file is available on Kaggle, but this session deliberately shows a shortcut: pandas can read a CSV directly from a URL, so you never need to download the file by hand.

```python
url = "https://docs.google.com/spreadsheets/d/...&output=csv"
df = pd.read_csv(url)
df.head()
```

`pd.read_csv(url)` performs the download and the parse in one call and returns a DataFrame; `df.head()` prints the first five rows so you can see the column names and a taste of the values. Read that output carefully — the exact spelling of every column you will filter, drop, or aggregate in Q-1 to Q-4 is defined right there.

Two settings were configured in the very first cell of the notebook: `pd.options.display.max_columns = None` tells pandas to show every column instead of truncating the middle ones, and the commented `pd.options.display.max_rows = None` would do the same for rows when you need the full table. On a dataset this wide, those display settings are what let you actually inspect the data.

From here on the pattern repeats through the whole notebook: import, read with read_csv, peek with head, then clean and question. Every later section follows this identical opening move.

In [ ]:
# To get the details about the dataset, please visit
# https://www.kaggle.com/datasets/sayanroy729/fifa-worldcup-2022-results

url = "https://docs.google.com/spreadsheets/d/e/2PACX-1vT3D_x_4DS6d51LKJ7ze1sxT5WpV5uiSVOFYHLwBiGru6vFyVv5h5-83AwFjxWYiWfCDjDAaarHAV-k/pub?gid=0&single=true&output=csv"
df = pd.read_csv(url)
df.head()

### Q-1: On-target percentage per team

Use the football dataset. Find out the total percentages that each team made on target. Display the result as a Python dictionary where the keys are the team list and the values are the percentage values. Round off the percentage values up to 2 decimal places.

The help block already gives the strategy; it is a filter-aggregate-structure recipe.

1. Get the unique team list. `df['Team'].unique()` returns every distinct team name — in this dataset each team appears roughly once per match it played. This list becomes the keys of your answer dictionary.

2. Loop over the teams. For each team, filter the rows where `df['Team'] == team`, then sum the total attempts column and the on-target column on those rows: `df[df['Team'] == team]['Attempts'].sum()` and the same for the on-target column.

3. Percentage = on-target sum divided by attempts sum, times 100. Build the dictionary with `dict[name] = round(percent, 2)`, giving values like 54.55 for Costa Rica.

4. Sort the dictionary BY VALUES — not keys — before printing. A plain dict has no order, so build the sorted view with `sorted(result.items(), key=lambda item: item[1], reverse=True)` and compare it with the sample block: every value rounded to two decimals, teams running from the best on-target percentage (Costa Rica, 54.55) down to Canada at 17.65.

> Hint: `unique()` for the team list, a for-loop filter, `sum()` twice, `round(..., 2)`, then sort items by value before printing.


**Sample Output:**
```bash
{'Costa Rica': 54.55,
 'Cameroon': 51.85,
 'Ecuador': 48.15,
 'Argentina': 46.99,
 'Brazil': 45.56,
 'England': 45.0,
 'Portugal': 40.32,
 'Ghana': 40.0,
 'Netherlands': 39.02,
 'Korea Republic': 36.73,
 'Australia': 36.0,
 'Mexico': 34.88,
 'Croatia': 34.78,
 'Germany': 34.33,
 'France': 32.97,
 'Spain': 32.69,
 'Belgium': 32.35,
 'Serbia': 32.26,
 'Iran': 31.43,
 'Uruguay': 31.25,
 'United States': 31.11,
 'Saudi Arabia': 31.03,
 'Senegal': 30.77,
 'Denmark': 30.56,
 'Switzerland': 30.56,
 'Japan': 30.23,
 'Wales': 29.17,
 'Qatar': 28.57,
 'Morocco': 28.3,
 'Tunisia': 26.67,
 'Poland': 25.0,
 'Canada': 17.65}
```

In [ ]:
# code here

### Q-2: Matches played and ranks

Find out how many times each team played in this Fifa World Cup 2022. On top of this, find out the ranks of the teams.

In this event-based dataset, the number of matches a team played is exactly how many rows carry that team in the `Team` column — so `df['Team'].value_counts()` is the plays-per-team table. The team that reached the final has the longest column of rows; teams that went out early have fewer.

Ranks attach a standing to each count. `value_counts()` returns a Series of match counts; calling `.rank()` on that Series ranks the teams from most matches to fewest. The note in the question is about the `method` parameter:

- default `method='average'` — ties share the average of their positions, producing fractional ranks like 2.5;
- `method='first'` — positions assigned in order of appearance, giving integer ranks (still returned as floats);
- `method='min'`, `method='max'`, and `method='dense'` — other tie policies.

So `df['Team'].value_counts().rank(method='first', ascending=False)` produces integer-looking ranks for every team, with the most-played team ranked 1. If you want the ranks ordered, chain `.sort_values()` or sort the underlying counts first and rank afterwards.

> Hint: `value_counts()` then `.rank(method='first', ascending=False)`; run it once without the method argument to see the 2.5-style fractional ranks.

In [ ]:
# code here

### Q-3: The cleaning pass

Find out the below topics:
- The information about the Fifa Worldcup dataset (`info()`).
- The description about the dataset (`describe()`).
- Check if there are any missing values; if there are, fill them with the average value for that particular column.
- Drop all duplicate rows permanently.
- Drop the columns: 'Sl No', 'Match No.', 'Red Cards' and 'Pts' permanently.

This is the classic data-quality checklist, and each bullet maps to one idea.

`df.info()` prints the column-by-column health chart — dtypes and non-null counts. `df.describe()` reports count, mean, std, min, percentiles, and max for every numeric column. Both together are the mandatory first look at any unfamiliar table.

Missing values: `df.isnull().sum()` counts the holes per column — the total number of cells needing attention. The fill is the tricky part: 'the average value for that particular column' means each column gets its own mean. For a purely numeric frame, `df.fillna(df.mean())` fills every column with its own column mean in one call — pandas aligns the columns automatically. If the frame contains text columns too, apply a per-column rule, for example `df = df.apply(lambda col: col.fillna(col.mean()) if col.dtype != 'object' else col)`.

Duplicates: `df.drop_duplicates(inplace=True)` deletes repeated full rows from the object permanently. Columns: `df.drop(columns=['Sl No', 'Match No.', 'Red Cards', 'Pts'], inplace=True)` removes exactly those four columns permanently. Order the steps sensibly — fill the holes, then drop duplicates and columns — and keep the operations on the same DataFrame object so the changes accumulate.

> Hint: info, describe, isnull().sum(), fillna with per-column means, drop_duplicates(inplace=True), drop(columns=[...], inplace=True).

In [ ]:
# code here

### Q-4: A Rank column and index

Do the below operations:
- Find out the rank based on the 'Team' column and save the result by adding a new column named 'Rank'.
- Change the datatype of this column to integer by using `np.int16`.
- Set the index of the DataFrame by using this 'Rank' column permanently.
- After that, sort the dataframe based on the 'Rank' index.

The four steps are a mini pipeline of column creation, dtype control, and index surgery.

Adding the column: `df['Rank'] = df['Team'].rank(method='first')` — pandas scans the Team column, assigns each row a rank (with `method='first'` the values stay integer-friendly), and writes the result into a brand-new column named Rank at the right edge of the frame.

Dtype: rank returns floats, but the notebook wants a compact integer. Convert with `df['Rank'] = df['Rank'].astype(np.int16)` — `np.int16` is the NumPy 16-bit signed integer type, exactly what the question names. `astype` is the general conversion tool: it rewrites the column's dtype, and because the assignment writes back into the same column name, the change is permanent.

Index surgery: `df.set_index('Rank', inplace=True)` promotes the Rank column to become the row labels. The column disappears as data and reappears on the left as the index.

Final sorting: `df.sort_index(inplace=True)` orders the rows by that new index, smallest Rank first. Since Rank is now the index, `sort_index` (by label) is the correct sort — `sort_values` sorts by a column instead.

> Hint: `df['Rank'] = ...`, then `.astype(np.int16)`, `set_index('Rank', inplace=True)`, `sort_index(inplace=True)`.

In [ ]:
# code here

# 5. The Titanic Dataset

The second dataset is the legendary Titanic passenger table — the same file used in the famous Kaggle competition. Each row is a passenger; the columns carry Name, Sex, Age, SibSp/Parch family counts, Ticket, Fare, Cabin, Embarked, and the target variable Survived. The raw competition file needs a Kaggle account to download; this task hands you two ready-to-read Google Sheets URLs instead, and `pd.read_csv(url)` loads either one directly.

The questions only care about the `Age` column and practise the two philosophies of missing-value handling on two different copies of the data:

- Dataset 1: drop rows. Remove the records that are missing Age, permanently.
- Dataset 2: fill rows. Patch the missing Age values with the constant 20, permanently.

The same technique you used on simpler frames applies here, scaled up to a real passenger table: spot holes with `isnull()`, delete with `dropna`, patch with `fillna`. Read the two files into two separate DataFrames so the two policies are exercised independently.

```python
url1 = "https://docs.google.com/spreadsheets/d/...gid=1562145139...&output=csv"
url2 = "https://docs.google.com/spreadsheets/d/...gid=1656109608...&output=csv"
df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
```

> Hint: keep the two frames separate — one gets dropna on Age, the other gets fillna(20) on Age.

### Q-5: Drop vs fill on the Age column

Do the below tasks:
1. With dataset 1, drop those records which only have missing values of the 'Age' column permanently.
2. With dataset 2, fill the missing values with 20 to the only 'Age' column permanently.

Part 1 is targeted deletion: only the Age column decides which rows leave. `df1.dropna(subset=['Age'], inplace=True)` — the `subset` argument tells pandas to judge a row's fate using just the Age column, so a passenger missing Cabin or Embarked survives as long as Age is present. With `inplace=True` the reduced frame is the one you keep.

Part 2 is targeted patching: every missing Age becomes 20. `df2['Age'].fillna(20, inplace=True)` reads as 'fill the Age holes with 20, permanently'. Passing a scalar broadcasts that constant to every gap; passing a Series (like `df2['Age'].mean()`) would instead stamp a statistic. After the fill, `df2['Age'].isnull().sum()` should report zero.

The pair of operations is the whole 'drop vs fill' decision in miniature. Dropping destroys rows and is appropriate when the holes are few or the column is unreliable; filling preserves every passenger and is appropriate when the column matters and a representative value is defensible. The choice of 20 is a deliberately simple constant — but the mechanism scales to mode-filling, mean-filling, and forward/backward filling.

> Hint: `df1.dropna(subset=['Age'], inplace=True)`; `df2['Age'].fillna(20, inplace=True)`; then verify with isnull().sum().

In [ ]:
# code here

# 6. The IPL Matches Dataset

The third dataset of the task is the full IPL match history from 2008 to 2022. A Google Sheets URL is provided that loads straight through `pd.read_csv`:

```python
ipl_matches = "https://docs.google.com/spreadsheets/d/...gid=1655759976...&output=csv"
matches_df = pd.read_csv(ipl_matches)
```

Each row is one match. The columns you will use in Q-6 to Q-10 include `Team1` and `Team2` (the two sides), `WinningTeam` and `Winner`, `Margin`, `Result` (with 'normal' and 'no result' style values), `City`, `Venue`, `Season` (values like '2022', '2021', '2020/21', '2009/10'), `MatchNumber` (with values like 'Final', 'Qualifier 2', 'Eliminator'), and the two string-packed columns `Team1Players` and `Team2Players`, which list a team's eleven players separated by commas.

Before ANY questions are attempted, the historical team-name changes must be applied, exactly as in the previous task:

```
'Delhi Daredevils' -> 'Delhi Capitals'
'Kings XI Punjab' -> 'Punjab Kings'
'Rising Pune Supergiants' -> 'Rising Pune Supergiant'
```

Q-6 states one extra rule that matters for the whole section: `Team1` represents the home team, and matches with no result must be excluded from the performance calculations. Keep those two facts in mind — every percentage question below depends on them.

> Hint: clean Team1 and Team2 with replace() before anything else, and filter out no-result matches where a question demands it.

### Q-6: Per-team performance table

Make a dataframe of each team in IPL with details like: Team Name, Matches Played, Win%, Home Win%, Away Win%. Show the dataframe sorted on Win%.

Replace old team names as new names before performing any tasks.

```
Delhi Daredevils -> Delhi Capitals
Kings XI Punjab -> Punjab Kings
Rising Pune Supergiants -> Rising Pune Supergiant
```

Note: Team1 represents the Home team. Exclude no-result matches.

This is a grouped summary problem — one row per team, several aggregate columns. The natural engine is `groupby`:

1. Clean first: `matches_df['Team1'] = matches_df['Team1'].replace({...})` and the same for `Team2`, mapping every historical name to its current form.
2. Exclude no results: `matches_df = matches_df[matches_df['Result'] != 'no result']` (the exact value depends on the file — inspect it). Now every remaining row is a decided match.
3. Matches played: a team's matches are their appearances in Team1 plus Team2 — `pd.concat([matches_df['Team1'], matches_df['Team2']]).value_counts()`.
4. Wins: count how often the team appears as the winning team, `matches_df['WinningTeam'].value_counts()`.
5. Percentages: Win% = wins / played * 100; Home Win% uses only matches as Team1, Away Win% only matches as Team2.

Assemble everything into a DataFrame with one team per row and the columns Team Name, Matches Played, Win%, Home Win%, Away Win%, then `sort_values('Win%', ascending=False)` so the strongest franchise tops the table.

> Hint: build per-team counts with groupby or value_counts, then Win% = wins / played * 100, with the Team1/Team2 split for home and away.

In [ ]:
# code here

### Q-7: Venues with most 'no result' matches

Some IPL matches end without a result — rain, mostly. Find the venues where that happens most often.

This is the two-step filter-then-count pattern. First isolate the abandoned matches: filter the frame to the rows whose result is 'no result' (how the file spells it is visible when you inspect `Result` — commonly 'no result'). Then count how many of those rows fall in each venue:

```python
no_result = matches_df[matches_df['Result'] == 'no result']
no_result['Venue'].value_counts()
```

`value_counts()` sorts descending, so the venue at the top of the output is the one that has suffered the most washouts. Use `.head(1)` to lock onto exactly the worst venue, or keep the full table to see the whole ranking.

Notice what did not need doing: no grouping, no joins — just a boolean mask on the whole frame and a frequency count on one column. The question is deliberately small after Q-6; it checks that you remember value_counts is the one-word answer to 'how many of each'.

> Hint: filter `Result`, then `['Venue'].value_counts()`.

In [ ]:
# code here

### Q-8: Most appearances in the final

Find the player with the most appearances in a final match.

The two columns `Team1Players` and `Team2Players` hold every player who took the field in a match — but not as a list. Each column is one long string per row, with player names separated by something like commas: 'MS Dhoni, Faf du Plessis, ...'. The hint in the question says it plainly: 'split and strip will help'.

The plan:

1. Isolate finals. Filter the frame to the rows that are the final — a `MatchNumber` (or Stage-like) column equal to 'Final'; inspect the file for the exact label.
2. Collect the players. Take the `Team1Players` and `Team2Players` values of those final rows and combine them so every player from both teams is in one place.
3. Split each string into names: `.str.split(',')` turns one string into a list of names, and `.str.strip()` removes the surrounding spaces. A clean approach: a Series of the player strings, `.str.split(',')`, then `explode()` and `.str.strip()` — or a loop that splits and strips row by row.
4. Count: `value_counts()` counts appearances; `value_counts().head(1)` names the player who played the most finals.

> Hint: combine the two player columns of the finals, split on ',', strip each name, then value_counts().head(1).

In [ ]:
# code here

### Q-9: IPL point table function

Make a function `point_table` which takes `season` as a parameter and shows the points table in non-ascending order of points and in ascending order of team name.

- For winning — 2 Points.
- For losing — 0 Points.
- For no result — both teams get 1 point.

Make a dataframe which will have `TeamName`, `MatchesPlayed`, `MatchesWon`, `NoResult`, `Points`, and make `TeamName` the index.

Season parameter should be one of: `['2022', '2021', '2020/21', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2009/10', '2009', '2007/08']`.

This is the capstone of the task — a complete, reusable analytics function. Dissect it into stages:

1. Filter to the season: `season_df = matches_df[matches_df['Season'] == season]`. This is why the whole IPL section started by cleaning team names.
2. Every team that played the season is a row of your answer table; the skeleton can come from `pd.concat([season_df['Team1'], season_df['Team2']]).unique()`.
3. Per team, count: MatchesPlayed — appearances as Team1 or Team2 in that season; MatchesWon — appearances in the winning-team column; NoResult — appearances in a no-result match. Remember the scoring: a no-result match gives BOTH teams a point, so that same NoResult count feeds Points directly.
4. Points = 2 * MatchesWon + 1 * NoResult. Losses need no column because Points already encodes the rule.
5. Assemble the DataFrame with TeamName as index, then `sort_values(['Points', 'TeamName'], ascending=[False, True])` — points descending first, then team name ascending as the tie-breaker. Compare your 2022 top two with the sample block: Gujarat Titans 16 played / 24 points, Rajasthan Royals 17 played / 20 points.

> Hint: build counts per team, then Points = 2 * Wins + NoResult; sort by Points desc then TeamName asc.

Output of two Top 2 in season 2022
```
TeamName    MatchesPlayed	MatchesWon	NoResult	Points
				
Gujarat Titans	    16	   12	       0	     24
Rajasthan Royals	  17	   10	       0	     20

```

In [ ]:
# code here

### Q-10: SeasonPosition column

Extend the above IPL point table with an extra column as `SeasonPosition`.

Teams below the top 4, after sorting on `Points` and then `TeamName`, will have the same `SeasonPosition` as their rank. Teams in the top four get special labels:

```
'Winner' - Team won the final
'Runner' - Team lost the final
3 - Losing team in Qualifier 2
4 - Losing team in Eliminator
```

For changing the value of a particular cell use `df.at[row_index, col_label] = value`.

If you try to change the value of a view of any dataframe, a warning will be shown. To avoid it, make a copy of the dataframe you want to change in, by `df.copy()`.

Two mechanisms are being practised here: `.rank()` for the bottom section and cell-by-cell assignment with `.at` for the top four. Work through it methodically.

1. Start from the point_table frame and `copy()` it — the note about the warning is the copy-versus-view lesson returning; editing a filtered view silences nothing, so snapshot first: `df = point_table('2022').copy()`.
2. Add the column with a rank on Points — for teams below the top four this is just their positional standing.
3. Identify the final, Qualifier 2, and Eliminator rows for that season from the matches frame (filter `Season` and the relevant `MatchNumber` values) and read off the losing team of each.
4. Overwrite the four special cells with `.at`: `df.at['Gujarat Titans', 'SeasonPosition'] = 'Winner'`, `df.at['Rajasthan Royals', 'SeasonPosition'] = 'Runner'`, and 3 / 4 for the losing teams of Qualifier 2 and Eliminator. `.at[label, col]` reaches a single cell by label, precisely the targeted edit the question wants. The final output shows every team, with the top two carrying Winner and Runner.

> Hint: `df.copy()` first, rank for the tail teams, then four `.at[label, col] = value` edits for the top-four labels.

## Summary

Task 18 was a full marathon of cleaning and analytics. On the FIFA World Cup data you computed per-team on-target percentages into a sorted dictionary, counted matches played and ranked the teams with `method='first'`, ran the whole data-quality pass — info, describe, missing counts, per-column mean filling, permanent duplicate and column drops — and turned a Rank column into the dataframe's index. On the Titanic files you applied the drop-vs-fill decision to the Age column with `dropna(subset=...)` and `fillna(20)`. Finally, on the IPL matches you cleaned renamed franchises, produced per-team win/home/away percentages, found the worst venues for no-result games, extracted the most frequent finalists from comma-packed player strings, and wrote a complete point_table(season) function extended by a SeasonPosition column using rank and .at cell edits. Every question was really the same cycle — read, clean, filter, count, aggregate — at increasing scale.

In [ ]:
# code here